# 04 — Forecast Model

Train a leakage-aware weekly demand model using lag, rolling, calendar, and promotion features. The model is evaluated on a chronological holdout and compared against the seasonal-naive baseline.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

def wape(actual,pred):
    denom=np.abs(actual).sum(); return np.abs(actual-pred).sum()/denom if denom else np.nan

p=Path('../data/analysis_ready_daily.csv')
if not p.exists(): raise FileNotFoundError('Run 01_Data_cleaning.ipynb first.')
daily=pd.read_csv(p, parse_dates=['date'])
weekly=(daily.groupby(['sku_id',pd.Grouper(key='date',freq='W-SUN')])
        .agg(units_sold=('units_sold','sum'), revenue=('revenue','sum'), promotion=('promotion','mean'), holiday_flag=('holiday_flag','max'))
        .reset_index().sort_values(['sku_id','date']))
weekly['month']=weekly.date.dt.month
weekly['weekofyear']=weekly.date.dt.isocalendar().week.astype(int)
for lag in [1,2,4,8]:
    weekly[f'lag_{lag}']=weekly.groupby('sku_id').units_sold.shift(lag)
weekly['rolling_mean_4']=weekly.groupby('sku_id').units_sold.transform(lambda s:s.shift(1).rolling(4).mean())
weekly['rolling_std_4']=weekly.groupby('sku_id').units_sold.transform(lambda s:s.shift(1).rolling(4).std())
weekly['season_sin']=np.sin(2*np.pi*weekly.weekofyear/52)
weekly['season_cos']=np.cos(2*np.pi*weekly.weekofyear/52)
weekly=weekly.dropna().reset_index(drop=True)
weekly.head()

## 1. Chronological train/test split

In [ ]:
cutoff=weekly.date.quantile(0.80)
train=weekly[weekly.date<=cutoff].copy(); test=weekly[weekly.date>cutoff].copy()
features=['lag_1','lag_2','lag_4','lag_8','rolling_mean_4','rolling_std_4','promotion','holiday_flag','month','weekofyear','season_sin','season_cos']
X_train=train[features]; y_train=train.units_sold
X_test=test[features]; y_test=test.units_sold
model=HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, max_leaf_nodes=15, random_state=42)
model.fit(X_train,y_train)
test['model_pred']=np.maximum(0, model.predict(X_test))
print('Train through:', train.date.max().date())
print('Test from:', test.date.min().date(), 'to', test.date.max().date())

## 2. Compare with seasonal-naive baseline

In [ ]:
test['baseline_pred']=test.groupby('sku_id').units_sold.shift(4)
# Shift within test is insufficient for the first four test weeks, so recover baseline from full history.
weekly_all=weekly[['sku_id','date','units_sold']].copy()
weekly_all['baseline_pred']=weekly_all.groupby('sku_id').units_sold.shift(4)
test=test.drop(columns=['baseline_pred']).merge(weekly_all[['sku_id','date','baseline_pred']],on=['sku_id','date'],how='left')
comparison=pd.DataFrame({
    'metric':['WAPE','MAE'],
    'seasonal_naive':[wape(test.units_sold,test.baseline_pred), mean_absolute_error(test.units_sold,test.baseline_pred)],
    'model':[wape(test.units_sold,test.model_pred), mean_absolute_error(test.units_sold,test.model_pred)]
})
display(comparison)

## 3. Backtest by forecast week

In [ ]:
weekly_scores=(test.groupby('date').apply(lambda g: pd.Series({
    'baseline_wape':wape(g.units_sold,g.baseline_pred),
    'model_wape':wape(g.units_sold,g.model_pred),
    'actual_units':g.units_sold.sum()
}), include_groups=False).reset_index())
display(weekly_scores)
print('Model beats baseline on overall WAPE:', comparison.loc[comparison.metric=='WAPE','model'].iloc[0] < comparison.loc[comparison.metric=='WAPE','seasonal_naive'].iloc[0])

## 4. Produce next-week SKU forecasts

This uses the most recent available weekly features. The output is a point forecast; prediction intervals can be added as a stretch goal.

In [ ]:
latest=weekly.sort_values('date').groupby('sku_id').tail(1).copy()
next_forecast=latest[['sku_id','date']+features].copy()
next_forecast['forecast_week']=next_forecast.date + pd.Timedelta(days=7)
next_forecast['forecast_units']=np.maximum(0, model.predict(next_forecast[features]))
next_forecast[['sku_id','forecast_week','forecast_units']].to_csv('../data/next_week_forecast.csv',index=False)
display(next_forecast[['sku_id','forecast_week','forecast_units']].head(20))

## 5. Save model evaluation

In [ ]:
comparison.to_csv('../data/forecast_model_comparison.csv',index=False)
weekly_scores.to_csv('../data/forecast_weekly_scores.csv',index=False)
print('Forecast artifacts saved.')